# MPECSS - MacMPEC Benchmark

This notebook runs the **MacMPEC** benchmark suite.

- **Dataset**: MacMPEC
- **Problems**: 191
- **Estimated time**: 4-6 hours
- **Resume support**: Built-in

Run the cells in order. After a Kaggle restart, re-run cells 1-3, then use the **Resume** cell.

## 1. Configuration

In [ ]:
# MacMPEC Configuration
import os

DATASET = 'macmpec'
RUN_TAG = 'Kaggle_MacMPEC'

WORKERS = 4
TIMEOUT = 3600
SAVE_LOGS = True
SHUFFLE = True

# Repository (code)
REPO_DIR = "/kaggle/working/MPECSSCODEPAPER"
REPO_GIT_URL = "https://github.com/mrsaurabhtanwar/MPECSSCODEPAPER.git"

# IMPORTANT: Results go directly to /kaggle/working/outputs to persist after session ends
OUTPUT_DIR = "/kaggle/working/outputs"

# Benchmark data bundled in the cloned repository
DATASET_PATH = os.path.join(REPO_DIR, "benchmarks", "macmpec", "macmpec-json")

# The repo is cloned in the next cell, so this path is checked there.
print(f"[INFO] Benchmark path: {DATASET_PATH}")

## 2. Clone Repository

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_DIR = Path(REPO_DIR)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

REPO_COMMIT = "SET_RELEASE_COMMIT"
if REPO_COMMIT == "SET_RELEASE_COMMIT":
    raise RuntimeError("Set REPO_COMMIT to the pushed release commit before the final Kaggle run.")
print(f"Cloning: {REPO_GIT_URL} @ {REPO_COMMIT}")
subprocess.run(["git", "clone", "--depth", "1", REPO_GIT_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "checkout", REPO_COMMIT], check=True, cwd=str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR))
print(f"[OK] Repository ready at: {REPO_DIR}")

## 3. Install Dependencies

In [ ]:
%%bash
cd /kaggle/working/MPECSSCODEPAPER
pip install -q --upgrade pip
pip install -q -e .
echo "[OK] Installation complete"

## 4. Verify Setup

In [ ]:
import os

print(f"Dataset path: {DATASET_PATH}")

if os.path.exists(DATASET_PATH):
    count = len([f for f in os.listdir(DATASET_PATH) if f.endswith('.json')])
    print(f"[OK] Found {count} problems")
else:
    print(f"[ERROR] Path not found: {DATASET_PATH}")

## 5. System Info

In [ ]:
import multiprocessing
import platform
import psutil

print("=" * 50)
print(f"Platform: {platform.platform()}")
print(f"Python: {platform.python_version()}")
print(f"CPUs: {multiprocessing.cpu_count()}")
mem = psutil.virtual_memory()
print(f"RAM: {mem.available / 1024**3:.1f} GB available / {mem.total / 1024**3:.1f} GB total")
print("=" * 50)

## 6. Preflight Checks

In [ ]:
%%bash
cd /kaggle/working/MPECSSCODEPAPER
python scripts/preflight_checks.py

## 7. Define Runner

In [ ]:
import subprocess
import sys

def run_benchmark(resume=False, summary=False):
    cmd = [
        sys.executable,
        f"{str(REPO_DIR)}/kaggle_setup/resumable_benchmark.py",
        "--dataset", DATASET,
        "--repo-dir", str(REPO_DIR),
        "--tag", RUN_TAG,
        "--workers", str(WORKERS),
        "--timeout", str(TIMEOUT),
        "--path", str(DATASET_PATH),
        "--output-dir", str(OUTPUT_DIR),  # CRITICAL: Save to persistent location
    ]
    if SAVE_LOGS: cmd.append("--save-logs")
    if SHUFFLE: cmd.append("--shuffle")
    if resume: cmd.append("--resume-latest")
    if summary: cmd.append("--summary-only")
    
    print("+ " + " ".join(str(x) for x in cmd))
    subprocess.run(cmd)

## 8. Run Benchmark (Fresh Start)

In [ ]:
run_benchmark()

## 9. Resume (After Restart)

In [ ]:
run_benchmark(resume=True)

## 10. Progress Summary

In [ ]:
run_benchmark(summary=True)

## 11. Copy Results

In [ ]:
from pathlib import Path
import shutil

output_dir = Path(OUTPUT_DIR)
if not output_dir.exists():
    print(f"[WARN] Output directory not found: {output_dir}")
else:
    archive = shutil.make_archive(str(output_dir), "zip", root_dir=str(output_dir))
    print(f"[OK] Results directory: {output_dir}")
    print(f"[OK] Download archive: {archive}")
    for item in sorted(output_dir.iterdir()):
        print(item)


## 12. Software Versions

In [ ]:
import casadi, numpy, scipy, pandas, platform
print(f"Python: {platform.python_version()}")
print(f"NumPy: {numpy.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"Pandas: {pandas.__version__}")
print(f"CasADi: {casadi.__version__}")